In [2]:
!pip install lightgbm

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.5 MB 162.9 kB/s eta 0:00:06
   -------------- ------------------------- 0.5/1.5 MB 162.9 kB/s eta 0:00:06
   -------------- ------------------------- 0.5/1.5 MB 162.9 kB/s eta 0:00:06
   -------------- ------------------------- 0.5/1.5 MB 162.9 kB/s eta 0:00:0

# Importing Libraries

In [3]:
import os, glob, warnings, copy
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing     import StandardScaler, LabelEncoder
from sklearn.model_selection   import train_test_split, StratifiedKFold
from sklearn.decomposition     import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline          import Pipeline

# Phase 2A — Binary classifiers
from sklearn.svm               import SVC
from sklearn.ensemble          import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neighbors         import KNeighborsClassifier
from sklearn.linear_model      import LogisticRegression
from sklearn.neural_network    import MLPClassifier
from xgboost                   import XGBClassifier
from lightgbm                  import LGBMClassifier

# Phase 2B — Anomaly detection
from sklearn.svm               import OneClassSVM
from sklearn.ensemble          import IsolationForest
from sklearn.neighbors         import LocalOutlierFactor
from sklearn.covariance        import EllipticEnvelope

from sklearn.metrics           import accuracy_score
from imblearn.over_sampling    import SMOTE

warnings.filterwarnings("ignore")

# Variable configurations

In [4]:
DATA_FOLDER      = "bbmas_data"        
RESULTS_FOLDER   = "results"
WINDOW_SIZE      = 30          # keystrokes per feature window (free-text has variable length)
WINDOW_STEP      = 15          # 50% overlap
MIN_WINDOWS      = 8           # minimum windows a user must have to be included
RECOMMENDED_USERS = None       # set to int (e.g. 50) to limit; None = use all valid users

os.makedirs(RESULTS_FOLDER, exist_ok=True)

# STEP 1: DATA LOADING AND WINDOWED FEATURE EXTRACTION

In [5]:
def load_processed_folder(folder: str, max_users: int = None) -> pd.DataFrame:
    """
    Load all *_processed.csv files and build a windowed feature matrix.
    Works even when users have different numbers of keystrokes.
    """
    pattern = os.path.join(folder, "*_processed.csv")
    files   = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(
            f"No *_processed.csv files found in '{folder}'.\n"
            f"Expected path: {os.path.abspath(folder)}"
        )
    if max_users:
        files = files[:max_users]

    all_windows = []
    skipped     = []

    # Core timing features always present
    CORE_FEATURES = ["press_to_press", "release_to_press", "hold_time"]
    # Optional extra features — used only if present in the file
    OPTIONAL_FEATURES = ["time_diff", "delta"]

    print(f"\n{'═'*60}")
    print(f"  Loading {len(files)} users from '{folder}'")
    print(f"  Window size: {WINDOW_SIZE} keystrokes  |  Step: {WINDOW_STEP}")
    print(f"{'═'*60}")

    for fp in files:
        uid = int(os.path.basename(fp).split("_")[0])
        df  = pd.read_csv(fp, low_memory=False)

        # Determine which features are available
        avail = CORE_FEATURES + [c for c in OPTIONAL_FEATURES if c in df.columns]
        df[avail] = df[avail].apply(pd.to_numeric, errors="coerce")
        df = df.dropna(subset=CORE_FEATURES)

        # IQR-based outlier removal per timing feature
        for col in CORE_FEATURES:
            Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
            IQR    = Q3 - Q1
            df = df[(df[col] >= Q1 - 2.5*IQR) & (df[col] <= Q3 + 2.5*IQR)]

        df = df.reset_index(drop=True)

        # Sliding window — produces one feature vector per window
        windows = extract_windows(df, avail, uid)

        if len(windows) < MIN_WINDOWS:
            skipped.append((uid, len(df), len(windows)))
            continue

        all_windows.extend(windows)
        print(f"  User {uid:3d}  |  {len(df):5d} keystrokes  |  {len(windows):4d} windows")

    if skipped:
        print(f"\n  ⚠  Skipped {len(skipped)} users (< {MIN_WINDOWS} windows):")
        for uid, rows, wins in skipped:
            print(f"      User {uid}: {rows} rows → {wins} windows")

    feature_df = pd.DataFrame(all_windows).fillna(0)
    print(f"\n  ✓ Feature matrix: {feature_df.shape[0]} windows × {feature_df.shape[1]-1} features")
    print(f"  ✓ Valid users   : {feature_df['user_id'].nunique()}")
    return feature_df

In [6]:
def extract_windows(df: pd.DataFrame, feature_cols: list, uid: int) -> list:
    """
    Slide a window of WINDOW_SIZE keystrokes with WINDOW_STEP overlap.
    Compute rich statistics per window — this is the 'feature vector'.
    """
    rows = []
    for start in range(0, len(df) - WINDOW_SIZE + 1, WINDOW_STEP):
        chunk = df.iloc[start:start + WINDOW_SIZE]
        feat  = {"user_id": uid}

        for col in feature_cols:
            s = chunk[col]
            feat[f"{col}_mean"]   = s.mean()
            feat[f"{col}_std"]    = s.std()
            feat[f"{col}_min"]    = s.min()
            feat[f"{col}_max"]    = s.max()
            feat[f"{col}_med"]    = s.median()
            feat[f"{col}_skew"]   = s.skew()        # asymmetry in timing
            feat[f"{col}_kurt"]   = s.kurt()        # peakedness
            feat[f"{col}_q25"]    = s.quantile(0.25)
            feat[f"{col}_q75"]    = s.quantile(0.75)
            feat[f"{col}_iqr"]    = feat[f"{col}_q75"] - feat[f"{col}_q25"]
            feat[f"{col}_cv"]     = (s.std() / (s.mean() + 1e-9))  # coeff. of variation

        # Cross-feature ratios — very discriminative for individuals
        ht_m  = feat.get("hold_time_mean",  1e-9)
        p2p_m = feat.get("press_to_press_mean", 1e-9)
        r2p_m = feat.get("release_to_press_mean", 1e-9)
        feat["ht_p2p_ratio"]  = ht_m  / (p2p_m + 1e-9)
        feat["ht_r2p_ratio"]  = ht_m  / (r2p_m + 1e-9)
        feat["p2p_r2p_ratio"] = p2p_m / (r2p_m + 1e-9)

        # Typing rhythm consistency — std of normalised intervals
        feat["rhythm_stability"] = chunk["press_to_press"].std() / (chunk["press_to_press"].mean() + 1e-9)

        rows.append(feat)
    return rows

# SHARED UTILITIES

In [7]:
def get_feature_cols(df: pd.DataFrame) -> list:
    return [c for c in df.columns if c != "user_id"]

In [8]:
def compute_far_frr_eer(y_true, y_scores):
    """Sweep thresholds → FAR, FRR, EER."""
    thresholds = np.linspace(0, 1, 300)
    far_list, frr_list = [], []
    for thresh in thresholds:
        y_pred = (y_scores >= thresh).astype(int)
        tp = np.sum((y_pred == 1) & (y_true == 1))
        fn = np.sum((y_pred == 0) & (y_true == 1))
        fp = np.sum((y_pred == 1) & (y_true == 0))
        tn = np.sum((y_pred == 0) & (y_true == 0))
        far_list.append(fp / (fp + tn + 1e-9))
        frr_list.append(fn / (fn + tp + 1e-9))

    far_arr = np.array(far_list)
    frr_arr = np.array(frr_list)
    eer_idx = np.argmin(np.abs(far_arr - frr_arr))
    eer     = (far_arr[eer_idx] + frr_arr[eer_idx]) / 2
    idx_05  = np.argmin(np.abs(thresholds - 0.5))
    return far_arr[idx_05], frr_arr[idx_05], eer

# STEP 2: PLOTTING

In [9]:
def make_plots(summary_df, detailed_df, metric_col, phase_tag):
    """Generate all 4 standard plots for a phase."""
    models = summary_df["Model"].tolist()

    # Plot 1 — Model comparison bar chart
    fig, ax = plt.subplots(figsize=(max(12, len(models)*2), 5))
    x    = np.arange(len(models))
    w    = 0.25
    cols = ["Avg FAR (%)", "Avg FRR (%)", "Avg EER (%)"]
    clrs = ["steelblue", "darkorange", "green"]
    for i, (col, clr) in enumerate(zip(cols, clrs)):
        ax.bar(x + i*w, summary_df[col], w, label=col, color=clr, alpha=0.85)
    ax.set_xticks(x + w)
    ax.set_xticklabels(models, rotation=20, ha="right", fontsize=9)
    ax.set_ylabel("Rate (%)")
    ax.set_title(f"{phase_tag} — Model Comparison (All Users)", fontsize=13, fontweight="bold")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_FOLDER, f"{phase_tag.lower().replace(' ','_')}_model_comparison.png"), dpi=150)
    plt.close()

    # Plot 2 — Per-user EER for best model
    best_model  = summary_df.sort_values("Avg EER (%)").iloc[0]["Model"]
    best_results = detailed_df[detailed_df["Model"] == best_model].sort_values("EER (%)")
    mean_eer     = best_results["EER (%)"].mean()
    plt.figure(figsize=(max(16, len(best_results)//3), 5))
    colors = ["steelblue" if e <= mean_eer else "salmon" for e in best_results["EER (%)"]]
    plt.bar(best_results["User"].astype(str), best_results["EER (%)"], color=colors, edgecolor="white")
    plt.axhline(mean_eer, color="red", linestyle="--", linewidth=1.5, label=f"Mean EER = {mean_eer:.2f}%")
    plt.title(f"{phase_tag} — Per-User EER ({best_model})", fontsize=13, fontweight="bold")
    plt.xlabel("User ID"); plt.ylabel("EER (%)")
    plt.xticks(rotation=90, fontsize=7)
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_FOLDER, f"{phase_tag.lower().replace(' ','_')}_per_user_eer.png"), dpi=150)
    plt.close()

    # Plot 3 — FAR vs FRR scatter (all models)
    n_models = len(models)
    fig, axes = plt.subplots(1, n_models, figsize=(4*n_models, 4), sharey=True)
    if n_models == 1: axes = [axes]
    palette = plt.cm.tab10.colors
    for ax, (model_name, clr) in zip(axes, zip(models, palette)):
        mdf = detailed_df[detailed_df["Model"] == model_name]
        ax.scatter(mdf["FAR (%)"], mdf["FRR (%)"], color=clr, alpha=0.7, edgecolors="white", s=50)
        ax.plot([0, 100], [0, 100], "k--", alpha=0.3, lw=1)
        ax.set_title(f"{model_name}\nEER={mdf['EER (%)'].mean():.2f}%", fontsize=8)
        ax.set_xlabel("FAR (%)"); ax.set_xlim(0, 100); ax.set_ylim(0, 100)
    axes[0].set_ylabel("FRR (%)")
    fig.suptitle(f"{phase_tag} — FAR vs FRR per User", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_FOLDER, f"{phase_tag.lower().replace(' ','_')}_far_frr_scatter.png"), dpi=150)
    plt.close()

    # Plot 4 — EER heatmap (users × models)
    try:
        eer_pivot = detailed_df.pivot_table(index="User", columns="Model", values="EER (%)", aggfunc="mean")
        eer_pivot = eer_pivot[[m for m in models if m in eer_pivot.columns]]
        h = max(8, len(eer_pivot) * 0.22)
        plt.figure(figsize=(max(10, n_models*2), h))
        sns.heatmap(eer_pivot, annot=(len(eer_pivot) <= 60), fmt=".1f",
                    cmap="RdYlGn_r", linewidths=0.2,
                    cbar_kws={"label": "EER (%)"},
                    annot_kws={"size": 6})
        plt.title(f"{phase_tag} — EER Heatmap (Green = better)", fontsize=12, fontweight="bold")
        plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_FOLDER, f"{phase_tag.lower().replace(' ','_')}_eer_heatmap.png"), dpi=150)
        plt.close()
    except Exception:
        pass


In [10]:
def print_summary(summary_df, phase_tag):
    w = 75
    print(f"\n{'═'*w}")
    print(f"  {phase_tag} — AVERAGED RESULTS ACROSS ALL USERS")
    print(f"{'═'*w}")
    print(f"{'Model':<30} {'Accuracy':>10} {'FAR (%)':>10} {'FRR (%)':>10} {'EER (%)':>10}")
    print(f"{'─'*w}")
    for _, r in summary_df.iterrows():
        acc = f"{r['Avg Accuracy (%)']:.2f}" if "Avg Accuracy (%)" in r else "  —  "
        print(f"{r['Model']:<30} {acc:>10} {r['Avg FAR (%)']:>10.2f} {r['Avg FRR (%)']:>10.2f} {r['Avg EER (%)']:>10.2f}")
    print(f"{'═'*w}")
    best = summary_df.sort_values("Avg EER (%)").iloc[0]
    print(f"  🏆 Best model → {best['Model']}  (EER = {best['Avg EER (%)']:.2f}%)")
    if "Avg Accuracy (%)" in best:
        print(f"     Avg Accuracy = {best['Avg Accuracy (%)']:.2f}%")
    print(f"     Avg FAR      = {best['Avg FAR (%)']:.2f}%")
    print(f"     Avg FRR      = {best['Avg FRR (%)']:.2f}%")

# PHASE 2A: BINARY CLASSIFICATION

In [11]:
def get_binary_classifiers():
    """
    Extended classifier set tuned for free-text keystroke biometrics.
    XGBoost and LightGBM added — both excel on tabular feature vectors.
    Extra Trees included for speed + diversity.
    MLP uses a deeper architecture than the CMU version.
    """
    return {
        "Random Forest": RandomForestClassifier(
            n_estimators=300, max_depth=None, min_samples_leaf=2,
            class_weight="balanced", n_jobs=-1, random_state=42),

        "XGBoost": XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            use_label_encoder=False, eval_metric="logloss",
            random_state=42, verbosity=0),

        "LightGBM": LGBMClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            num_leaves=63, subsample=0.8, colsample_bytree=0.8,
            class_weight="balanced", random_state=42, verbose=-1),

        "Extra Trees": ExtraTreesClassifier(
            n_estimators=300, class_weight="balanced",
            n_jobs=-1, random_state=42),

        "SVM (RBF)": SVC(
            kernel="rbf", C=10, gamma="scale",
            probability=True, class_weight="balanced", random_state=42),

        "MLP Neural Net": MLPClassifier(
            hidden_layer_sizes=(128, 64, 32), activation="relu",
            max_iter=500, early_stopping=True, validation_fraction=0.1,
            random_state=42),

        "KNN": KNeighborsClassifier(n_neighbors=7, n_jobs=-1),

        "Logistic Regression": LogisticRegression(
            C=1.0, max_iter=1000, class_weight="balanced", random_state=42),
    }

In [12]:
def build_binary_dataset(feature_df, feature_cols, target_uid):
    """
    Genuine windows for target_uid = class 1
    Equal number of windows sampled from all other users = class 0
    Handles class imbalance with SMOTE on training split.
    """
    genuine  = feature_df[feature_df["user_id"] == target_uid][feature_cols].copy()
    impostor = feature_df[feature_df["user_id"] != target_uid][feature_cols].copy()

    n_imp = min(len(genuine) * 3, len(impostor))  # up to 3:1 impostor ratio
    impostor = impostor.sample(n=n_imp, random_state=42)

    genuine["label"]  = 1
    impostor["label"] = 0

    combined = pd.concat([genuine, impostor], ignore_index=True)
    X = combined[feature_cols].values
    y = combined["label"].values
    return X, y

In [13]:
def run_phase2a(feature_df: pd.DataFrame):
    """Phase 2A — Binary classification loop over all users × all models."""
    feature_cols = get_feature_cols(feature_df)
    users        = sorted(feature_df["user_id"].unique())
    classifiers  = get_binary_classifiers()

    print(f"\n{'═'*60}")
    print(f"  PHASE 2A — BINARY CLASSIFICATION")
    print(f"  Users: {len(users)}  |  Features: {len(feature_cols)}  |  Models: {len(classifiers)}")
    print(f"{'═'*60}\n")

    model_acc  = {n: [] for n in classifiers}
    model_far  = {n: [] for n in classifiers}
    model_frr  = {n: [] for n in classifiers}
    model_eer  = {n: [] for n in classifiers}
    detailed   = []

    for idx, uid in enumerate(users, 1):
        X, y = build_binary_dataset(feature_df, feature_cols, uid)

        if len(np.unique(y)) < 2 or np.sum(y == 1) < 6:
            print(f"  [{idx:03d}] User {uid:3d}  ⚠  Skipped — insufficient genuine samples")
            continue

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, stratify=y, random_state=42)

        # Scale
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s  = scaler.transform(X_test)

        # SMOTE — balance classes in training set
        try:
            sm = SMOTE(random_state=42, k_neighbors=min(3, np.sum(y_train==1)-1))
            X_train_s, y_train = sm.fit_resample(X_train_s, y_train)
        except Exception:
            pass  # skip SMOTE if not enough neighbours

        print(f"  [{idx:03d}/{len(users)}] User {uid:3d}  genuine={np.sum(y==1):3d}  total={len(y)}", end="")

        for name, clf in classifiers.items():
            m = copy.deepcopy(clf)
            m.fit(X_train_s, y_train)
            y_scores = m.predict_proba(X_test_s)[:, 1]
            acc      = accuracy_score(y_test, (y_scores >= 0.5).astype(int))
            far, frr, eer = compute_far_frr_eer(y_test, y_scores)

            model_acc[name].append(acc)
            model_far[name].append(far)
            model_frr[name].append(frr)
            model_eer[name].append(eer)
            detailed.append({
                "User": uid, "Model": name,
                "Accuracy (%)": round(acc*100, 2),
                "FAR (%)":  round(far*100, 2),
                "FRR (%)":  round(frr*100, 2),
                "EER (%)":  round(eer*100, 2),
            })
        print("  ✓")

    # Summary
    summary_rows = []
    for name in classifiers:
        if not model_eer[name]: continue
        summary_rows.append({
            "Model":              name,
            "Avg Accuracy (%)":   round(np.mean(model_acc[name])*100, 2),
            "Avg FAR (%)":        round(np.mean(model_far[name])*100, 2),
            "Avg FRR (%)":        round(np.mean(model_frr[name])*100, 2),
            "Avg EER (%)":        round(np.mean(model_eer[name])*100, 2),
        })

    summary_df  = pd.DataFrame(summary_rows).sort_values("Avg EER (%)").reset_index(drop=True)
    detailed_df = pd.DataFrame(detailed)

    print_summary(summary_df, "Phase 2A")
    summary_df.to_csv(os.path.join(RESULTS_FOLDER, "phase2a_summary.csv"),  index=False)
    detailed_df.to_csv(os.path.join(RESULTS_FOLDER, "phase2a_detailed.csv"), index=False)
    make_plots(summary_df, detailed_df, "EER (%)", "Phase 2A")
    print(f"\n  📄 Saved: results/phase2a_summary.csv & phase2a_detailed.csv")

    return summary_df, detailed_df

# PHASE 2B: ANOMALY DETECTION

In [14]:
def tune_ocsvm(X_train_pca):
    """Find best nu for One-Class SVM via internal FRR minimisation."""
    best_nu, best_frr = 0.05, 1.0
    X_t, X_v = train_test_split(X_train_pca, test_size=0.2, random_state=42)
    for nu in [0.001, 0.005, 0.01, 0.02, 0.05, 0.1]:
        clf = OneClassSVM(kernel="rbf", nu=nu)
        clf.fit(X_t)
        frr = np.mean(clf.predict(X_v) == -1)
        if frr < best_frr:
            best_frr = frr
            best_nu  = nu
    return best_nu

In [15]:
def scores_from_anomaly(model, X):
    """Convert decision_function output to 0-1 probability-like scores."""
    raw = model.decision_function(X)
    # Normalise: higher score = more genuine
    mn, mx = raw.min(), raw.max()
    if mx == mn:
        return np.full(len(raw), 0.5)
    return (raw - mn) / (mx - mn)

In [16]:
def run_phase2b(feature_df: pd.DataFrame):
    """Phase 2B — One-class / anomaly detection loop."""
    feature_cols = get_feature_cols(feature_df)
    users        = sorted(feature_df["user_id"].unique())

    print(f"\n{'═'*60}")
    print(f"  PHASE 2B — ANOMALY DETECTION")
    print(f"  Users: {len(users)}  |  Features: {len(feature_cols)}")
    print(f"{'═'*60}\n")

    model_far = {"One-Class SVM": [], "Isolation Forest": [], "Local Outlier Factor": [], "Elliptic Envelope": []}
    model_frr = {k: [] for k in model_far}
    model_eer = {k: [] for k in model_far}
    detailed  = []

    for idx, uid in enumerate(users, 1):
        genuine_df  = feature_df[feature_df["user_id"] == uid][feature_cols].copy()
        impostor_df = feature_df[feature_df["user_id"] != uid][feature_cols].copy()

        if len(genuine_df) < MIN_WINDOWS:
            print(f"  [{idx:03d}] User {uid:3d}  ⚠  Skipped")
            continue

        # IQR outlier removal on genuine set
        Q1, Q3 = genuine_df.quantile(0.25), genuine_df.quantile(0.75)
        IQR    = Q3 - Q1
        mask   = ((genuine_df < Q1 - 2*IQR) | (genuine_df > Q3 + 2*IQR)).sum(axis=1)
        genuine_clean = genuine_df[mask <= 3]

        X_g_train, X_g_test = train_test_split(genuine_clean, test_size=0.2, random_state=42)

        # Sample same number of impostors as genuine test
        n_imp   = min(len(X_g_test) * 5, len(impostor_df))
        X_imp   = impostor_df.sample(n=n_imp, random_state=42)

        scaler  = StandardScaler()
        X_tr_s  = scaler.fit_transform(X_g_train)
        X_gt_s  = scaler.transform(X_g_test)
        X_im_s  = scaler.transform(X_imp)

        # PCA — retain 95% variance
        pca     = PCA(n_components=0.95, random_state=42)
        X_tr_p  = pca.fit_transform(X_tr_s)
        X_gt_p  = pca.transform(X_gt_s)
        X_im_p  = pca.transform(X_im_s)

        # Variance threshold post-PCA
        sel     = VarianceThreshold(threshold=0.005)
        X_tr_p  = sel.fit_transform(X_tr_p)
        X_gt_p  = sel.transform(X_gt_p)
        X_im_p  = sel.transform(X_im_p)

        best_nu = tune_ocsvm(X_tr_p)

        models = {
            "One-Class SVM":        OneClassSVM(kernel="rbf", nu=best_nu),
            "Isolation Forest":     IsolationForest(n_estimators=300, contamination=0.05, random_state=42),
            "Local Outlier Factor": LocalOutlierFactor(n_neighbors=min(20, len(X_tr_p)-1), contamination=0.05, novelty=True),
            "Elliptic Envelope":    EllipticEnvelope(contamination=0.05, random_state=42),
        }

        print(f"  [{idx:03d}/{len(users)}] User {uid:3d}  genuine={len(genuine_clean):3d}  PCA={X_tr_p.shape[1]}  nu={best_nu}", end="")

        for name, model in models.items():
            try:
                model.fit(X_tr_p)
                # Genuine: +1 = accepted, -1 = rejected
                g_pred = model.predict(X_gt_p)
                i_pred = model.predict(X_im_p)
                FRR    = np.mean(g_pred == -1)
                FAR    = np.mean(i_pred ==  1)
                EER    = (FAR + FRR) / 2

                model_far[name].append(FAR)
                model_frr[name].append(FRR)
                model_eer[name].append(EER)
                detailed.append({
                    "User": uid, "Model": name,
                    "FAR (%)": round(FAR*100, 2),
                    "FRR (%)": round(FRR*100, 2),
                    "EER (%)": round(EER*100, 2),
                })
            except Exception as e:
                pass
        print("  ✓")

    summary_rows = []
    for name in model_far:
        if not model_eer[name]: continue
        summary_rows.append({
            "Model":        name,
            "Avg FAR (%)":  round(np.mean(model_far[name])*100, 2),
            "Avg FRR (%)":  round(np.mean(model_frr[name])*100, 2),
            "Avg EER (%)":  round(np.mean(model_eer[name])*100, 2),
        })

    summary_df  = pd.DataFrame(summary_rows).sort_values("Avg EER (%)").reset_index(drop=True)
    detailed_df = pd.DataFrame(detailed)

    print_summary(summary_df, "Phase 2B")
    summary_df.to_csv(os.path.join(RESULTS_FOLDER, "phase2b_summary.csv"),  index=False)
    detailed_df.to_csv(os.path.join(RESULTS_FOLDER, "phase2b_detailed.csv"), index=False)
    make_plots(summary_df, detailed_df, "EER (%)", "Phase 2B")
    print(f"\n  📄 Saved: results/phase2b_summary.csv & phase2b_detailed.csv")

    return summary_df, detailed_df

# USER COUNT RECOMMENDATION

In [17]:
def print_user_recommendation(feature_df):
    """
    Analyse how many users are needed for stable model evaluation.
    Rule of thumb for continuous authentication research:
        ≥ 30  users → results start being statistically stable
        ≥ 50  users → publication-quality baselines
        ≥ 100 users → strong generalisability claim
    """
    n = feature_df["user_id"].nunique()
    wins = feature_df.groupby("user_id").size()
    print(f"\n{'═'*60}")
    print(f"  USER COUNT ANALYSIS")
    print(f"{'═'*60}")
    print(f"  Valid users loaded : {n}")
    print(f"  Windows per user   : min={wins.min()}  max={wins.max()}  mean={wins.mean():.1f}")
    print(f"\n  RECOMMENDATION:")
    if n >= 100:
        print(f" {n} users — Excellent. Full dataset gives strong generalisation.")
    elif n >= 50:
        print(f" {n} users — Good for publication-quality results.")
        print(f"  Consider adding more users if available.")
    elif n >= 30:
        print(f" {n} users — Acceptable. Results may have higher variance.")
        print(f"  Recommend ≥ 50 users for reliable EER estimates.")
    else:
        print(f" {n} users — Too few. Strongly recommend ≥ 30 users.")
    print(f"{'═'*60}")

# MAIN PIPELINE

In [18]:
if __name__ == "__main__":
    print("\n" + "█"*60)
    print("  BBMAS FREE-TEXT KEYSTROKE AUTHENTICATION PIPELINE")
    print("  Phase 2A: Binary Classification")
    print("  Phase 2B: Anomaly Detection")
    print("█"*60)

    # ── Load & Build Feature Matrix ───────────────────────────
    feature_df = load_processed_folder(DATA_FOLDER, max_users=RECOMMENDED_USERS)

    print_user_recommendation(feature_df)

    # ── Phase 2A ──────────────────────────────────────────────
    summary_2a, detailed_2a = run_phase2a(feature_df)

    # ── Phase 2B ──────────────────────────────────────────────
    summary_2b, detailed_2b = run_phase2b(feature_df)

    # ── Final Cross-Phase Summary ─────────────────────────────
    print(f"\n{'█'*60}")
    print(f"  FINAL RECOMMENDATION FOR DEPLOYMENT")
    print(f"{'█'*60}")
    best_2a = summary_2a.iloc[0]
    best_2b = summary_2b.iloc[0]
    print(f"\n  Phase 2A Best → {best_2a['Model']}")
    print(f"    Accuracy = {best_2a['Avg Accuracy (%)']:.2f}%  EER = {best_2a['Avg EER (%)']:.2f}%")
    print(f"\n  Phase 2B Best → {best_2b['Model']}")
    print(f"    EER = {best_2b['Avg EER (%)']:.2f}%")
    deploy = best_2a['Model'] if best_2a['Avg EER (%)'] <= best_2b['Avg EER (%)'] else best_2b['Model']
    phase  = "2A (Binary)" if best_2a['Avg EER (%)'] <= best_2b['Avg EER (%)'] else "2B (Anomaly)"
    print(f"\n  DEPLOY → {deploy}  [{phase}]")
    print(f"\n  All results + plots saved to: ./{RESULTS_FOLDER}/")
    print("█"*60 + "\n")


████████████████████████████████████████████████████████████
  BBMAS FREE-TEXT KEYSTROKE AUTHENTICATION PIPELINE
  Phase 2A: Binary Classification
  Phase 2B: Anomaly Detection
████████████████████████████████████████████████████████████

════════════════════════════════════════════════════════════
  Loading 116 users from 'bbmas_data'
  Window size: 30 keystrokes  |  Step: 15
════════════════════════════════════════════════════════════
  User 100  |   1490 keystrokes  |    98 windows
  User 101  |   1103 keystrokes  |    72 windows
  User 102  |   1314 keystrokes  |    86 windows
  User 103  |   1517 keystrokes  |   100 windows
  User 104  |   1058 keystrokes  |    69 windows
  User 105  |   1462 keystrokes  |    96 windows
  User 106  |   1414 keystrokes  |    93 windows
  User 107  |   1212 keystrokes  |    79 windows
  User 108  |   1761 keystrokes  |   116 windows
  User 109  |   1599 keystrokes  |   105 windows
  User  10  |   1523 keystrokes  |   100 windows
  User 110  |   130